# Module 10.1: Quantization Fundamentals

Welcome to Module 10! You've built a full Transformer, but as you scale it up, you hit a brutal reality: **VRAM (Video RAM) is expensive and scarce.**

A 70 Billion parameter model in 16-bit precision requires ~140GB of VRAM just to load the weights (let alone context and activations!). 
How can we run these models on consumer GPUs? The answer is **Quantization**: shrinking the neural network's parameters by squeezing high-precision floating-point numbers into low-precision integer formats (like 8-bit or 4-bit).

## 1. Absolute Maximum (Absmax) Quantization

### The Concept
A standard `float16` or `float32` number can represent highly precise fractional values over a massive range. An `int8` (8-bit integer) can only represent 256 discrete whole numbers: from `-128` to `127`.

To squeeze our weights into `int8`, we need a mapping strategy. **Absmax Quantization** is symmetric. We find the absolute maximum value in our weight tensor, say `3.2`, and we define that as our maximum integer `127`. Everything else is scaled proportionally.

### The Math
$$ Scale = \frac{127}{max(|W|)} $$
$$ W_{quantized} = round(W \times Scale) $$
$$ W_{dequantized} = \frac{W_{quantized}}{Scale} $$

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)  # Reproducible results

def absmax_quantize(tensor):
    # 1. Find the absolute maximum value
    absmax = torch.max(torch.abs(tensor))
    
    # 2. Calculate the scaling factor
    # 127 is the max positive value an int8 can hold
    scale = 127.0 / absmax
    
    # 3. Scale the tensor and round to nearest integer
    quantized = torch.round(tensor * scale)
    
    # 4. Cast safely to int8 (clamping just to be safe against rounding errors)
    quantized_int8 = torch.clamp(quantized, -128, 127).to(torch.int8)
    return quantized_int8, scale

def absmax_dequantize(quantized_tensor, scale):
    # Cast back to float32 before dividing, and divide by the exact same scale
    return quantized_tensor.to(torch.float32) / scale

## 2. Let's see the Loss of Precision

Quantization is a "lossy" compression. By forcing floating point numbers into integer buckets, we lose the tiny fractional details. Let's visualize this.

In [ ]:
# Create a mock weight matrix with standard normal distribution (like real neural weights)
original_weights = torch.randn(10, 10)

quantized_weights, scale = absmax_quantize(original_weights)
recovered_weights = absmax_dequantize(quantized_weights, scale)

print(f"Scale Factor: {scale:.4f}")
print(f"Original Byte Size: {original_weights.element_size() * original_weights.nelement()} bytes as Float32")
print(f"Quantized Byte Size: {quantized_weights.element_size() * quantized_weights.nelement()} bytes as Int8 (We saved 75% memory!)")

# Calculate the Error (Mean Squared Error)
mse_error = torch.nn.functional.mse_loss(original_weights, recovered_weights)
print(f"\nPrecision Loss (MSE): {mse_error.item():.6f}")

# Look at a single specific weight to understand rounding error
print(f"\nOriginal weight value: {original_weights[0, 0].item():.4f}")
print(f"Recovered weight value: {recovered_weights[0, 0].item():.4f}")

## 3. Zero-Point (Asymmetric) Quantization

### The Problem with Absmax
Absmax assumes your neural network weights are symmetric around zero. But what if all your weights are between `10.0` and `20.0`? If you use Absmax, the max is `20.0`, mapped to `127`. The value `-127` would represent `-20.0`. But you have NO negative weights! Half of your precious `int8` buckets (`-128` to `0`) are completely wasted.

### The Solution: Zero-Point
Instead of mapping the max absolute value, we find the overall `min` and `max` of our tensor, and map that exact range to `0` and `255` (the range of `uint8`). We calculate a `zero_point` that shifts the numbers so the smallest weight lands on bucket `0`.

### Why is the `zero_point` rounded to an integer?
The `zero_point` is itself a **stored quantized code** — it is the integer bucket that the float value `0.0` maps to. Because dequantization computes `(q - zero_point) / scale`, both `q` and `zero_point` must be integers for the arithmetic to stay in integer space on the hardware. Rounding it to an integer guarantees that the float `0.0` maps *exactly* onto a real bucket (no fractional bucket exists), so a true zero in the weights survives the round-trip without drift.

### A worked numeric example
Suppose a tensor's range is exactly `[10.0, 20.0]` and we map it onto `uint8` `[0, 255]`:
- `scale = 255 / (20.0 - 10.0) = 25.5`
- `zero_point = round(-10.0 * 25.5) = round(-255) = -255`

Now quantize the endpoints with `q = round(value * scale) + zero_point`:
- value `10.0` → `round(10.0 * 25.5) + (-255) = 255 - 255 = 0`  ✅ (lowest bucket)
- value `20.0` → `round(20.0 * 25.5) + (-255) = 510 - 255 = 255` ✅ (highest bucket)

And dequantize back with `(q - zero_point) / scale`:
- `0` → `(0 - (-255)) / 25.5 = 255 / 25.5 = 10.0` ✅
- `255` → `(255 - (-255)) / 25.5 = 510 / 25.5 = 20.0` ✅

Every one of the 256 buckets is now used to cover only the range we care about — nothing is wasted on negative numbers that never appear.

In [ ]:
def asymmetric_quantize(tensor):
    t_min, t_max = tensor.min(), tensor.max()
    
    # Map the range (max - min) onto 255 (the range of uint8)
    scale = 255.0 / (t_max - t_min)
    
    # The zero_point is the integer bucket that float 0.0 maps to.
    # It MUST be an integer because dequant does (q - zero_point) / scale,
    # and both q and zero_point are stored integer codes on the hardware.
    zero_point = torch.round(-t_min * scale).to(torch.int32)
    
    # Quantize, shift by zero_point, and clip into [0, 255]
    quantized = torch.clamp(torch.round(tensor * scale) + zero_point, 0, 255).to(torch.uint8)
    
    return quantized, scale, zero_point

def asymmetric_dequantize(quantized_tensor, scale, zero_point):
    # Order of operations matters: shift back by zero_point, then divide by scale
    return (quantized_tensor.to(torch.float32) - zero_point) / scale


# Test with purely positive weights (where Absmax wastes half its buckets)
positive_weights = torch.rand(10, 10) * 10 + 10  # Random floats from 10.0 to 20.0

# --- Asymmetric path ---
q_asym, scale_asym, zp_asym = asymmetric_quantize(positive_weights)
recovered_asym = asymmetric_dequantize(q_asym, scale_asym, zp_asym)
mse_asym = torch.nn.functional.mse_loss(positive_weights, recovered_asym).item()

# --- Absmax path on the SAME tensor (so we can compare apples to apples) ---
q_abs, scale_abs = absmax_quantize(positive_weights)
recovered_abs = absmax_dequantize(q_abs, scale_abs)
mse_abs = torch.nn.functional.mse_loss(positive_weights, recovered_abs).item()

print("Quantizing the SAME positive tensor (range ~10.0 to 20.0) two ways:\n")
print(f"  Absmax (symmetric)      MSE: {mse_abs:.8f}")
print(f"  Asymmetric (zero-point) MSE: {mse_asym:.8f}")
print(f"\nAsymmetric is ~{mse_abs / mse_asym:.1f}x more accurate here, because Absmax")
print("wastes every bucket below 127 on negative values that never appear.")

## 4. Modern Quantization (LLM.int8(), GPTQ, AWQ)

If it's this simple, why are there so many papers?

- **Outlier Features**: In massive LLMs, some hidden dimensions suddenly spike to huge values (e.g., `100.0`). If you quantize the whole matrix, that single `100.0` will force the `scale` so low that all normal weights get crushed into the `int8` value `0`. Papers like **LLM.int8()** solve this by keeping outliers in `float16` and quantizing the rest.
- **Group Quantization**: Instead of finding one `Scale` for the whole matrix, we find a separate `Scale` for every chunk of 128 numbers. This retains vastly more precision.
- **4-Bit**: People have successfully pushed weights down to 4 bits using Non-Linear Quantization (like NF4 in QLoRA), drastically reducing VRAM usage further.

In [ ]:
# Starter code — fill in the TODOs

# --- Task 1: INT4 absmax ---
def absmax_quantize_int4(tensor):
    absmax = torch.max(torch.abs(tensor))
    scale = 7.0 / absmax                       # INT4 signed max is 7
    quantized = torch.round(tensor * scale)
    quantized = torch.clamp(quantized, -8, 7)  # 4-bit signed range
    return quantized, scale

def absmax_dequantize_simple(q, scale):
    return q.to(torch.float32) / scale

w = torch.randn(10, 10)
# TODO: quantize w with both absmax_quantize (INT8) and absmax_quantize_int4,
#       dequantize each, and print the two MSEs side by side.


# --- Task 2: outlier crushing ---
outlier_tensor = torch.randn(64).clone()
outlier_tensor[0] = 50.0  # one giant outlier
# TODO: quantize the whole tensor at once vs. in groups of 16 (separate scale each),
#       then print recovered_tensor[1:5] for both to SEE the normal weights survive
#       under group quantization but get crushed under per-tensor quantization.


### 🏋️ Try it yourself

1. **INT4 instead of INT8.** Write `absmax_quantize_int4(tensor)` that maps onto the 4-bit signed range `[-8, 7]` (use `7.0 / absmax` as the scale and clamp to `[-8, 7]`). Quantize a `torch.randn(10, 10)` tensor with both INT8 and your INT4 version and compare the MSEs. How much worse does precision get when you go from 16 buckets to 256?

2. **See the outlier "crushing" effect.** Take a normal tensor but inject a single huge outlier, then quantize it (a) as one whole tensor and (b) in small *groups* with a separate scale per group. Watch the per-tensor version crush the normal weights toward zero while group quantization protects them.